In [ ]:
# Cell 1: Imports

import mailbox
import email
import re
import json 
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup 
from tqdm.notebook import tqdm


PROJECT = Path.home() / "llm-mail-trainer"
MBOX_FILE = PROJECT / "data/raw/All mail Including Spam and Trash-002.mbox"

print(f"MBOX file exists: {MBOX_FILE.exists()}")
print(f"File size: {MBOX_FILE.stat().st_size / 1e9:.2f} GB")

# Cell 2: Open MBOX and count emails

mbox = mailbox.mbox(str(MBOX_FILE))
total_emails = len(mbox)
print(f"Total emails in mailbox: {total_emails:,}")

# Cell 3: Look at one raw email
sample = mbox[0]

print("===EMAIL KEYS===")
print(list(sample.keys()))

print("\n=== SUBJECT ===")
print(sample['subject'])

print("\n=== FROM ===")
print(sample['from'])

print("\n=== DATE ===")
print(sample['date'])

print("\n=== CONTENT TYPE ===")
print(sample.get_content_type())

# Cell 4: Extract body from email

def get_body(message):
    """Extract text from email body."""

    if message.is_multipart():
        # Email has multiple parts (text + html + attachments)
        for part in message.walk():
            ctype = part.get_content_type()
            if ctype == 'text/plain':
                payload = part.get_payload(decode=True)
                if payload:
                    return payload.decode('utf-8', errors='ignore')
            elif ctype == 'text/html':
                payload = part.get_payload(decode=True)
                if payload:
                    soup = BeautifulSoup(payload.decode('utf-8', errors='ignore'),'lxml')
                    return soup.get_text(separator=' ',strip= True)
    else:
        # Single part email
        payload = message.get_payload(decode=True)
        if payload:
            text = payload.decode('utf-8', errors='ignore')
            if message.get_content_type() == 'text/html':
                soup = BeautifulSoup(text, 'lxml')
                return soup.get_text(separator=' ',strip= True)
            return text

    return ''

# Test on sample email
body = get_body(sample)
print(f"Body length: {len(body)} characters")
print(f"\n=== FIRST 500 CHARS ===\n{body[:500]}")

# Cell 5: Clean text function

def clean_text(text):
    """Remove noise from email text."""

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove very long strings (encoded data)
    text = re.sub(r'\S{80,}', '', text)
    
    return text.strip()

# Test
cleaned = clean_text(body)
print(f"Before cleaning: {len(body)} chars")
print(f"After cleaning: {len(cleaned)} chars")
print(f"\n=== CLEANED TEXT ===\n{cleaned}")

# Cell 6: Decode encoded headers (like Subject, From)
def decode_header(header):
    """Decode email header that may be encoded."""
    if header is None:
        return ''
    
    try:
        decoded_parts = email.header.decode_header(header)
        result = []
        for content, charset in decoded_parts:
            if isinstance(content, bytes):
                content = content.decode(charset or 'utf-8', errors='ignore')
            result.append(str(content))
        return ' '.join(result)
    except Exception:
        return str(header)

# Test on the HDFC email
hdfc_email = mbox[27]

print("=== DECODED SUBJECT ===")
print(decode_header(hdfc_email['subject']))

print("\n=== DECODED FROM ===")
print(decode_header(hdfc_email['from']))

# Cell 7: Complete single email parser
def parse_email(message):
    """Parse a single email into clean structured data."""

    body = get_body(message)
    body = clean_text(body)

     # Skip if body too short
    if len(body) < 50:
        return None

    return {
        'subject' : clean_text(decode_header(message['subject'])),
        'sender' : clean_text(decode_header(message['from'])),
        'date': message['date'] or '',
        'body' : body[:5000]
        
    }

# Test on HDFC email
result = parse_email(hdfc_email)

print("=== PARSED EMAIL ===")
for key, value in result.items():
    if key == 'body':
        print(f"{key}: {value[:200]}...")
    else:
        print(f"{key}: {value}")

# Cell 8: Parse all emails
parsed_emails = []
failed = 0

print("Parsing all the mails ...")
for i in tqdm(range(total_emails), desc="Processing"):
    try:
        msg = mbox[i]
        result = parse_email(msg)
        if result:
            result['id'] = len(parsed_emails)
            parsed_emails.append(result)
    except Exception as e:
        failed += 1
        continue

print(f"\n✅ Successfully parsed: {len(parsed_emails):,}")
print(f"❌ Failed/skipped: {failed + (total_emails - len(parsed_emails) - failed):,}")
print(f"📊 Success rate: {len(parsed_emails)/total_emails*100:.1f}%")

# Cell 9: Save parsed emails to JSON
output_path = PROJECT / "data/parsed/emails.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(parsed_emails, f, ensure_ascii=False)

# Verify
file_size = output_path.stat().st_size / 1e6
print(f"✅ Saved to: {output_path}")
print(f"📁 File size: {file_size:.1f} MB")
print(f"📧 Total emails: {len(parsed_emails):,}")

# Cell 10: Data summary
import pandas as pd

df = pd.DataFrame(parsed_emails)

print("=== DATA SUMMARY ===")
print(f"Total emails: {len(df):,}")
print(f"\nColumns: {list(df.columns)}")

print(f"\n=== BODY LENGTH STATS ===")
df['body_length'] = df['body'].str.len()
print(df['body_length'].describe())

print(f"\n=== TOP 10 SENDERS ===")
print(df['sender'].value_counts().head(10))